In [ ]:
!python --version

In [ ]:
import os
from huggingface_hub import login

hf_token = "TOKEN"

os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

login(token=hf_token)





In [ ]:
!git clone  https://huggingface.co/spaces/pragnakalp/OCR-image-to-text

In [ ]:
%cd OCR-image-to-text/

In [ ]:
!apt-get update
!apt-get install -y git-lfs
!git lfs install

!pip install --upgrade pip setuptools wheel

!pip install gradio==3.50.2 datasets==2.16.1 huggingface-hub==0.22.2 easyocr==1.7.1 paddleocr paddlepaddle opencv-python pillow numpy pandas matplotlib tensorflow

In [ ]:
!apt-get install git-lfs
!git lfs install


In [ ]:
!pip install -r requirements.txt


In [ ]:
!python app.py

In [ ]:
!pip install langchain-text-splitters

In [ ]:
import gradio as gr
import tensorflow as tf

import requests
import cv2
import os
import csv
import numpy as np
import pandas as pd
import huggingface_hub
from huggingface_hub import Repository
from datetime import datetime
import scipy.ndimage.interpolation as inter
import easyocr
import datasets
from datasets import load_dataset, Image
from paddleocr import PaddleOCR
from save_data import flag
from PIL import Image as PILImage


def ocr_with_paddle(img):
    ocr = PaddleOCR(lang='en', use_angle_cls=True)
    result = ocr.ocr(img)
    return ' '.join([line[1][0] for line in result[0]]) if result and result[0] else ''

def ocr_with_keras(img):
    pipeline = keras_ocr.pipeline.Pipeline()
    images = [keras_ocr.tools.read(img)]
    predictions = pipeline.recognize(images)
    return ' '.join([text for text, box in predictions[0]])

def ocr_with_easy(img):
    reader = easyocr.Reader(['en'])
    bounds = reader.readtext(img, paragraph="False", detail=0)
    return ' '.join(bounds)

def load_hieroglyph_image(char, style, size=(250, 250)):
    image_path = f"hieroglyphics/{style}/{char}.png"
    try:
        image = PILImage.open(image_path)
        resized_image = image.resize(size)
        return resized_image
    except FileNotFoundError:
        print(f"File not found: {image_path}")
        return None

def concatenate_images(images):
    if not images or any(img is None for img in images):
        return None
    total_width = sum(img.width for img in images)
    max_height = max(img.height for img in images)
    new_img = PILImage.new('RGB', (total_width, max_height))
    x_offset = 0
    for img in images:
        if img:
            new_img.paste(img, (x_offset, 0))
            x_offset += img.width
    return new_img

def translate_to_hieroglyphs(english_text, style):
    chars = [char for char in english_text.lower() if char.isalnum()]
    images = [load_hieroglyph_image(char, style) for char in chars]
    return concatenate_images(images)

# Define a dictionary mapping English letters to Hieroglyphic Unicode characters
hieroglyphic_mapping = {
    'a': '\U0001313F', 'b': '\U000130C0', 'c': '\U000133A1', 'd': '\U000130A7',
    'e': '\U000131CB', 'f': '\U00013191', 'g': '\U000133BC', 'h': '\U00013254',
    'i': '\U000131CB', 'j': '\U00013193', 'k': '\U000133A1', 'l': '\U000130ED',
    'm': '\U00013153', 'n': '\U00013216', 'o': '\U0001336F', 'p': '\U0001342F',
    'q': '\U0001320E', 'r': '\U0001308B', 's': '\U000132F4', 't': '\U000133CF',
    'u': '\U00013171', 'v': '\U00013191', 'w': '\U00013171', 'x': '\U00013121',
    'y': '\U000131CC', 'z': '\U00013283', ' ': ' '
}

def translate_to_hieroglyphic_text(english_text):
    return ''.join(hieroglyphic_mapping.get(char, char) for char in english_text.lower())

def generate_ocr(method, img, style):
    if img.any():
        if method == 'EasyOCR':
            text_output = ocr_with_easy(img)
        elif method == 'KerasOCR':
            text_output = ocr_with_keras(img)
        else:
            text_output = ocr_with_paddle(img)

        hieroglyph_image = translate_to_hieroglyphs(text_output, style)
        hieroglyph_text = translate_to_hieroglyphic_text(text_output)
        return hieroglyph_image, hieroglyph_text
    else:
        raise gr.Error("Please upload an image!")

image = gr.Image()
method = gr.Radio(["PaddleOCR", "EasyOCR", "KerasOCR"], value="PaddleOCR")
style = gr.Dropdown(["Style1", "Style2", "Style3","Style4"], label="Choose Style", value="Style1")
output_image = gr.Image(label="Hieroglyphs Image")
output_text = gr.Textbox(label="Translated Text")

demo = gr.Interface(
    fn=generate_ocr,
    inputs=[method, image, style],
    outputs=[output_image, output_text],
    title="Optical Character Recognition and Hieroglyphic Translation"
)

demo.launch(share=True)


In [ ]:
import gradio as gr
import easyocr
from PIL import Image as PILImage
import numpy as np

reader = easyocr.Reader(['en'])

def ocr_with_easy(img):
    result = reader.readtext(img, detail=0)
    return ' '.join(result)

def load_hieroglyph_image(char, style, size=(250, 250)):
    image_path = f"hieroglyphics/{style}/{char}.png"
    try:
        image = PILImage.open(image_path)
        return image.resize(size)
    except FileNotFoundError:
        print(f"File not found: {image_path}")
        return None

def concatenate_images(images):
    images = [img for img in images if img is not None]

    if len(images) == 0:
        return None

    total_width = sum(img.width for img in images)
    max_height = max(img.height for img in images)

    new_img = PILImage.new('RGB', (total_width, max_height), "white")

    x_offset = 0
    for img in images:
        new_img.paste(img, (x_offset, 0))
        x_offset += img.width

    return new_img

def translate_to_hieroglyphs(english_text, style):
    chars = [char for char in english_text.lower() if char.isalnum()]
    images = [load_hieroglyph_image(char, style) for char in chars]
    return concatenate_images(images)

hieroglyphic_mapping = {
    'a': '\U0001313F', 'b': '\U000130C0', 'c': '\U000133A1', 'd': '\U000130A7',
    'e': '\U000131CB', 'f': '\U00013191', 'g': '\U000133BC', 'h': '\U00013254',
    'i': '\U000131CB', 'j': '\U00013193', 'k': '\U000133A1', 'l': '\U000130ED',
    'm': '\U00013153', 'n': '\U00013216', 'o': '\U0001336F', 'p': '\U0001342F',
    'q': '\U0001320E', 'r': '\U0001308B', 's': '\U000132F4', 't': '\U000133CF',
    'u': '\U00013171', 'v': '\U00013191', 'w': '\U00013171', 'x': '\U00013121',
    'y': '\U000131CC', 'z': '\U00013283', ' ': ' '
}

def translate_to_hieroglyphic_text(english_text):
    return ''.join(hieroglyphic_mapping.get(char, char) for char in english_text.lower())

def generate_ocr(img, style):
    if img is None:
        raise gr.Error("Please upload an image.")

    text_output = ocr_with_easy(img)

    hieroglyph_image = translate_to_hieroglyphs(text_output, style)
    hieroglyph_text = translate_to_hieroglyphic_text(text_output)

    return text_output, hieroglyph_image, hieroglyph_text

image = gr.Image(label="Upload Image")
style = gr.Dropdown(
    ["Style1", "Style2", "Style3", "Style4"],
    label="Choose Style",
    value="Style1"
)

output_text_ocr = gr.Textbox(label="Detected English Text")
output_image = gr.Image(label="Hieroglyphs Image")
output_text = gr.Textbox(label="Hieroglyphic Unicode Text")

demo = gr.Interface(
    fn=generate_ocr,
    inputs=[image, style],
    outputs=[output_text_ocr, output_image, output_text],
    title="OCR Image to Hieroglyphic Translator"
)

demo.launch(share=True, debug=True)

In [ ]:
# from PIL import Image
# import matplotlib.pyplot as plt

# # Function to load a hieroglyphic image based on the character
# def load_hieroglyph_image(char, size=(150, 100)):
#     # Load Hieroglyphic character image based on the given char (e.g., 'a.png', 'b.png')
#     image_path = f"hieroglyphics//{char}.png"  # Adjust the path to your actual hieroglyphic images directory
#     image = Image.open(image_path)

#     resized_image = image.resize(size)
#     return resized_image


# # Function to concatenate hieroglyphic images horizontally into a single image
# def concatenate_images(image_list):
#     widths, heights = zip(*(i.size for i in image_list))
#     total_width = sum(widths)
#     max_height = max(heights)
#     new_image = Image.new('RGB', (total_width, max_height))
#     x_offset = 0
#     for img in image_list:
#         new_image.paste(img, (x_offset, 0))
#         x_offset += img.width
#     return new_image

# # English text to be translated
# style = "1"
# english = "caty"
# english_text= english.lower()

# # Translation logic (replace with your translation method)
# translated_text = [char for char in english_text]

# # Load hieroglyphic images and concatenate them
# hieroglyph_images = [load_hieroglyph_image(char) for char in translated_text]
# concatenated_image = concatenate_images(hieroglyph_images)

# # Display the concatenated image
# plt.imshow(concatenated_image)
# plt.axis('off')
# plt.show()
from PIL import Image
import matplotlib.pyplot as plt

# Function to load a hieroglyphic image based on the character and style
def load_hieroglyph_image(char, style, size=(150, 100)):
    # Construct the file path including the style directory
    image_path = f"hieroglyphics/{style}/{char}.png"  # Update the path format to include the style directory
    image = Image.open(image_path)

    resized_image = image.resize(size)
    return resized_image

# Function to concatenate hieroglyphic images horizontally into a single image
def concatenate_images(image_list):
    widths, heights = zip(*(i.size for i in image_list))
    total_width = sum(widths)
    max_height = max(heights)
    new_image = Image.new('RGB', (total_width, max_height))
    x_offset = 0
    for img in image_list:
        new_image.paste(img, (x_offset, 0))
        x_offset += img.width
    return new_image

# User inputs for translation
style = "2"  # Style directory name
english = "caty"  # Text to translate
english_text = english.lower()

# Translation logic (simply a list of characters for this example)
translated_text = [char for char in english_text]

# Load hieroglyphic images for each character according to the specified style and concatenate them
hieroglyph_images = [load_hieroglyph_image(char, style) for char in translated_text]
concatenated_image = concatenate_images(hieroglyph_images)

# Display the concatenated image
plt.imshow(concatenated_image)
plt.axis('off')
plt.show()
